# Proyecto #2: Pipeline de Machine Learning
## Notebook 03: Creación del pipeline de ingeniería de características
### Predict Future Sales (Kaggle)

**Objetivo:** convertir las decisiones del notebook 02 en un **pipeline de sklearn** con los pasos de ingeniería de características (operadores de `feature-engine` y el operador `Mapper` visto en clase), ajustarlo con los meses de entrenamiento y guardarlo como `models/preprocessing_pipeline.pkl`.

**Salida:** `models/preprocessing_pipeline.pkl`, que el notebook 04 usará para transformar los datos y al que le agregará el modelo ganador como paso final.

## 1. Los lags se construyen antes del pipeline

Como se explicó en el notebook 02, los pasos de un pipeline de sklearn transforman **cada fila por separado**, mientras que un lag necesita el **historial** del par tienda-producto (las ventas de los meses anteriores), que no viene en la fila a predecir: `test.csv` solo tiene `ID`, `shop_id` e `item_id`. Si los lags se calcularan dentro del pipeline, el pipeline tendría que guardar todo `sales_train` y, al predecir el mes 34, mezclar filas de distintos meses, algo para lo que la API `fit`/`transform` no está pensada.

Por eso el proceso tiene dos partes:

1. **`build_feature_matrix`** (`predict_future_sales/features.py`): construye la tabla mensual tienda x producto con el objetivo, los lags 1, 2, 3, 6 y 12, los lags agregados y las variables de catálogo e historia. Es la **misma función** que usan los notebooks 02, 04 y 05, así que entrenamiento, validación y test se construyen con el mismo código y sin fuga de información (cada lag del mes *m* usa solo meses anteriores a *m*).
2. **El pipeline de este notebook:** recibe esa tabla (sin el objetivo) y aplica los seis procesos de ingeniería de características. Es lo que se guarda en el `.pkl`.

El operador `Mapper` (unificación de tiendas duplicadas) aparece en las dos partes: dentro de `build_feature_matrix` se aplica a las ventas **antes** de calcular los lags, para que la historia de las tiendas 0, 1 y 11 se sume a la de las tiendas 57, 58 y 10; y es el **primer paso** del pipeline, para que cualquier dato que llegue con los IDs antiguos se traduzca al ID vigente antes de codificar.

In [1]:
import gc
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from feature_engine.encoding import OneHotEncoder, OrdinalEncoder
from feature_engine.imputation import ArbitraryNumberImputer, CategoricalImputer, MeanMedianImputer
from feature_engine.outliers import Winsorizer
from feature_engine.selection import DropFeatures
from feature_engine.transformation import LogCpTransformer, LogTransformer
from feature_engine.wrappers import SklearnTransformerWrapper
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from predict_future_sales import features as F
from predict_future_sales.dataset import load_raw
from predict_future_sales.transformers import DowncastFloat32, Mapper

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.4f}".format)
MODELS_DIR = Path("../models")

## 2. Construcción de la tabla de modelado y división secuencial

Se usa la división secuencial del enunciado: **entrenamiento con los meses 0 a 26 y validación con los meses 27 a 33**. Como las filas empiezan en el mes 12 (primer mes con lag de 12 meses), el entrenamiento efectivo son los meses **12 a 26**; los meses 0 a 11 solo aportan historia para los lags. El pipeline se ajusta **únicamente con el entrenamiento**.

In [2]:
raw = load_raw()
inicio = time.time()
matriz = F.build_feature_matrix(raw)
print(f"Tabla construida en {time.time() - inicio:,.0f} s: {len(matriz):,} filas x {matriz.shape[1]} columnas, "
      f"{matriz.memory_usage(deep=True).sum() / 1e6:,.0f} MB")

X_train, y_train = F.split_xy(matriz, range(F.FIRST_MODEL_MONTH, F.LAST_TRAIN_MONTH + 1))
X_valid, y_valid = F.split_xy(matriz, range(F.LAST_TRAIN_MONTH + 1, F.TEST_MONTH))
X_test, _ = F.split_xy(matriz, [F.TEST_MONTH])
del matriz
gc.collect()

for nombre, X in [("Entrenamiento (12-26)", X_train), ("Validación (27-33)", X_valid), ("Test (34)", X_test)]:
    print(f"{nombre:<22} {X.shape[0]:>10,} filas | meses {X['date_block_num'].min()}-{X['date_block_num'].max()}")
print("\nColumnas de entrada del pipeline:", X_train.columns.tolist())

Tabla construida en 29 s: 6,639,294 filas x 21 columnas, 345 MB


Entrenamiento (12-26)   4,810,730 filas | meses 12-26
Validación (27-33)      1,614,364 filas | meses 27-33
Test (34)                 214,200 filas | meses 34-34

Columnas de entrada del pipeline: ['date_block_num', 'shop_id', 'item_id', 'month', 'item_category_id', 'shop_city', 'item_cat_type', 'item_cat_subtype', 'item_cnt_month_lag_1', 'item_cnt_month_lag_2', 'item_cnt_month_lag_3', 'item_cnt_month_lag_6', 'item_cnt_month_lag_12', 'item_avg_cnt_lag_1', 'shop_avg_cnt_lag_1', 'shop_cat_avg_cnt_lag_1', 'item_price_last', 'item_age', 'item_is_new', 'shop_item_is_new']


**Interpretación:**

- `build_feature_matrix` construye la tabla en menos de un minuto y ocupa unos **345 MB** con los 5 lags elegidos y los tipos reducidos.
- El **entrenamiento** tiene 4,810,730 filas (meses 12 a 26), la **validación** 1,614,364 (meses 27 a 33) y el **test** 214,200 (mes 34, los pares de `test.csv`). La validación tiene menos filas por mes que el entrenamiento porque la red de tiendas y el catálogo activo se reducen con el tiempo, en línea con la tendencia decreciente del EDA.
- La entrada del pipeline son 20 columnas: las llaves (`date_block_num`, `shop_id`, `item_id`), las variables de catálogo, los 5 lags del par, los 3 lags agregados, el precio previo y las 3 variables de historia del producto. El objetivo `item_cnt_month` se separa como `y`.

### Tienda y producto como variables exógenas

El enunciado pide que el modelo prediga `item_cnt_month` considerando la **tienda** y el **producto** como variables exógenas o banderas, es decir, como información externa a la propia serie de ventas que ayuda a explicarla. Además de los lags del par, el modelo recibe:

**Tienda:**
- `shop_id`, que entra **directamente** como variable, codificada con `OrdinalEncoder(encoding_method="ordered")`: cada tienda recibe un código según su venta media en el entrenamiento (la tienda 31, la más grande del EDA, recibe el código más alto). Así el modelo distingue el nivel de cada sucursal.
- `shop_city`, codificada de la misma forma, para compartir información entre tiendas de una misma ciudad.
- `shop_avg_cnt_lag_1` y `shop_cat_avg_cnt_lag_1`: el nivel de ventas reciente de la tienda y de cada categoría dentro de ella.

**Producto:**
- `item_category_id` (codificada por venta media), el subtipo `item_cat_subtype` (codificado igual) y el tipo `item_cat_type` (6 banderas one-hot): describen qué clase de producto es.
- `item_avg_cnt_lag_1`: las ventas del producto en todas las tiendas el mes anterior, junto con los lags 1, 2, 3, 6 y 12 del par tienda-producto.
- `item_price_last`: el último precio conocido del producto.
- `item_age` y las banderas `item_is_new` (producto sin ventas previas) y `shop_item_is_new` (producto que nunca se vendió en esa tienda).

**Por qué se elimina `item_id` como número crudo:** `item_id` es una etiqueta arbitraria, no una cantidad. Si entrara como número, el modelo supondría un orden y una distancia que no existen (el producto 5000 no se parece más al 5001 que al 20000), y un árbol podría dividir rangos de IDs sin sentido. Tampoco se puede codificar: con unos 22,000 productos, un one-hot sería inmanejable en memoria, y una codificación por venta media sería muy ruidosa para los miles de productos con pocas ventas. Además, los 363 productos nuevos de test no tienen ningún valor aprendido. Por eso la identidad del producto se representa con su **historial de ventas, su categoría, su precio y su antigüedad**, que sí existen (o se imputan) para cualquier producto, incluidos los nuevos.

## 3. Definición del pipeline

Los pasos siguen el orden y los criterios definidos en la sección 4 del notebook 02:

| # | Nombre del paso | Proceso | Operador |
|---|---|---|---|
| 1 | `unificar_tiendas` | Mapeo | `Mapper` ({0: 57, 1: 58, 11: 10}) |
| 2 | `imputar_lags` | Imputación numérica | `ArbitraryNumberImputer(0)` |
| 3 | `imputar_precio` | Imputación numérica | `MeanMedianImputer("median")` |
| 4 | `imputar_categoricas` | Imputación categórica | `CategoricalImputer("frequent")` |
| 5 | `codificar_ordinal` | Codificación | `OrdinalEncoder("ordered")` |
| 6 | `codificar_onehot` | Codificación | `OneHotEncoder(top_categories=6)` |
| 7 | `eliminar_item_id` | Selección | `DropFeatures(["item_id"])` |
| 8 | `outliers_precio` | Outliers | `Winsorizer("iqr", fold=3)` |
| 9 | `log_precio` | Transformación | `LogTransformer` |
| 10 | `log_conteos` | Transformación | `LogCpTransformer(C=1)` |
| 11 | `escalar` | Escalado | `SklearnTransformerWrapper(StandardScaler())` |
| 12 | `float32` | Memoria | `DowncastFloat32` |

Las listas de variables se toman de las constantes de `predict_future_sales/features.py`, las mismas que usa `build_feature_matrix`, para que el pipeline y la tabla no puedan desincronizarse.

In [3]:
preprocesamiento = Pipeline([
    # 1. Mapeo: tiendas duplicadas hacia el ID que aparece en test (EDA, sección 4.c)
    ("unificar_tiendas", Mapper(variables=["shop_id"], mappings=F.SHOP_MAPPINGS)),
    # 2-3. Imputación numérica: NaN en un lag = sin ventas -> 0; precio desconocido -> mediana
    ("imputar_lags", ArbitraryNumberImputer(arbitrary_number=0, variables=F.count_lag_features())),
    ("imputar_precio", MeanMedianImputer(imputation_method="median", variables=F.PRICE_FEATURES)),
    # 4. Imputación categórica: ciudad de las tiendas no físicas y subtipo de categorías especiales
    ("imputar_categoricas", CategoricalImputer(imputation_method="frequent", variables=["shop_city", "item_cat_subtype"])),
    # 5-6. Codificación: tienda y categoría como variables exógenas
    ("codificar_ordinal", OrdinalEncoder(
        encoding_method="ordered",
        variables=["shop_id", "item_category_id", "shop_city", "item_cat_subtype"],
        ignore_format=True,
        unseen="encode",
    )),
    ("codificar_onehot", OneHotEncoder(top_categories=6, variables=["item_cat_type"])),
    # 7. Se elimina el identificador del producto
    ("eliminar_item_id", DropFeatures(features_to_drop=["item_id"])),
    # 8. Outliers del precio (cola derecha, criterio IQR)
    ("outliers_precio", Winsorizer(capping_method="iqr", tail="right", fold=3, variables=F.PRICE_FEATURES)),
    # 9-10. Transformaciones logarítmicas
    ("log_precio", LogTransformer(variables=F.PRICE_FEATURES)),
    ("log_conteos", LogCpTransformer(variables=F.count_lag_features(), C=1)),
    # 11. Escalado de todas las columnas
    ("escalar", SklearnTransformerWrapper(transformer=StandardScaler())),
    # 12. float32 para reducir memoria
    ("float32", DowncastFloat32()),
])
preprocesamiento

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('unificar_tiendas', ...), ('imputar_lags', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,variables,['shop_id']
,mappings,"{0: 57, 1: 58, 11: 10}"
,arbitrary_number,0
,variables,"['item_cnt_month_lag_1', 'item_cnt_month_lag_2', ...]"
,imputer_dict,None
,variables,['item_price_last']
,imputation_method,'median'


## 4. Ajuste del pipeline con los meses de entrenamiento

El pipeline recibe también `y_train`, porque el `OrdinalEncoder` con `encoding_method="ordered"` necesita el objetivo para ordenar las categorías según su venta media. Los demás pasos lo ignoran.

In [4]:
inicio = time.time()
X_train_t = preprocesamiento.fit_transform(X_train, y_train)
print(f"Ajuste y transformación del entrenamiento: {time.time() - inicio:,.1f} s")
print(f"Resultado: {X_train_t.shape[0]:,} filas x {X_train_t.shape[1]} columnas | "
      f"NaN: {X_train_t.isna().sum().sum()} | tipos: {X_train_t.dtypes.astype(str).unique().tolist()} | "
      f"memoria: {X_train_t.memory_usage().sum() / 1e6:,.0f} MB")
print("\nColumnas de salida:", X_train_t.columns.tolist())

Ajuste y transformación del entrenamiento: 11.2 s
Resultado: 4,810,730 filas x 24 columnas | NaN: 0 | tipos: ['float32'] | memoria: 462 MB

Columnas de salida: ['date_block_num', 'shop_id', 'month', 'item_category_id', 'shop_city', 'item_cat_subtype', 'item_cnt_month_lag_1', 'item_cnt_month_lag_2', 'item_cnt_month_lag_3', 'item_cnt_month_lag_6', 'item_cnt_month_lag_12', 'item_avg_cnt_lag_1', 'shop_avg_cnt_lag_1', 'shop_cat_avg_cnt_lag_1', 'item_price_last', 'item_age', 'item_is_new', 'shop_item_is_new', 'item_cat_type_Кино', 'item_cat_type_Музыка', 'item_cat_type_Подарки', 'item_cat_type_Игры PC', 'item_cat_type_Игры', 'item_cat_type_Книги']


In [5]:
# Parámetros aprendidos por cada paso (solo con los meses 12 a 26)
pasos = preprocesamiento.named_steps
print("Mapper:", pasos["unificar_tiendas"].mappings)
print("Valor imputado en los lags:", pasos["imputar_lags"].imputer_dict_["item_cnt_month_lag_1"])
print("Mediana del precio:", pasos["imputar_precio"].imputer_dict_)
print("Moda de las categóricas:", pasos["imputar_categoricas"].imputer_dict_)
print("Categorías del one-hot:", pasos["codificar_onehot"].encoder_dict_)
print("Límite superior del precio:", pasos["outliers_precio"].right_tail_caps_)

codigos = pasos["codificar_ordinal"].encoder_dict_
print("\nCategorías codificadas por variable:", {k: len(v) for k, v in codigos.items()})
print("Código de algunas tiendas (ordenado por venta media):",
      {t: codigos["shop_id"][t] for t in [36, 34, 57, 42, 25, 31] if t in codigos["shop_id"]})

escala = pasos["escalar"].transformer_
print("\nMedia y desviación aprendidas por el StandardScaler (primeras 6 variables):")
display(pd.DataFrame({"media": escala.mean_, "desviación": escala.scale_}, index=pasos["escalar"].variables_).head(6))

Mapper: {0: 57, 1: 58, 11: 10}
Valor imputado en los lags: 0
Mediana del precio: {'item_price_last': 299.0}
Moda de las categóricas: {'shop_city': 'Москва', 'item_cat_subtype': 'DVD'}
Categorías del one-hot: {'item_cat_type': ['Кино', 'Музыка', 'Подарки', 'Игры PC', 'Игры', 'Книги']}
Límite superior del precio: {'item_price_last': 2201.857177734375}

Categorías codificadas por variable: {'shop_id': 53, 'item_category_id': 75, 'shop_city': 28, 'item_cat_subtype': 60}
Código de algunas tiendas (ordenado por venta media): {34: 0, 57: 46, 42: 45, 25: 51, 31: 52}

Media y desviación aprendidas por el StandardScaler (primeras 6 variables):


,media,desviación
date_block_num,18.7966,4.2886
shop_id,26.1393,14.7448
month,5.6234,3.5945
item_category_id,36.0654,13.6917
shop_city,16.1227,8.5482
item_cat_subtype,29.0762,9.6319


**Interpretación del ajuste:**

- El pipeline completo se ajusta y transforma los 4.8 millones de filas en unos **10 segundos**, porque todos los pasos son operaciones vectorizadas (descartar Yeo-Johnson en el notebook 02 evitó unos 5 minutos de ajuste).
- La salida tiene **24 columnas, todas `float32`, sin ningún NaN**, y ocupa 462 MB. Las 20 columnas de entrada se convierten en 24: se elimina `item_id`, `item_cat_type` se reemplaza por 6 banderas one-hot y el resto se conserva transformado.
- Los parámetros aprendidos son los mismos que en el notebook 02: 0 para los lags, mediana de 299 para el precio, Москва y DVD como modas, límite de 2,201.86 para el precio y los 6 tipos de categoría más frecuentes para el one-hot.
- El `OrdinalEncoder` aprendió un diccionario por variable (53 tiendas, 75 categorías, 28 ciudades y 60 subtipos). El código de la tienda refleja su nivel de ventas: la 34 recibe 0, la 57 (Якутск) 46, la 25 (Москва ТРК "Атриум") 51 y la 31 (Москва ТЦ "Семеновский", la de mayor venta del EDA) 52. Así la tienda entra al modelo como variable exógena con un orden que tiene sentido.
- El `StandardScaler` guarda la media y la desviación de cada columna; por ejemplo, la media de `date_block_num` en el entrenamiento es 18.8 y su desviación 4.3.

## 5. Verificaciones del pipeline

### 5.a El Mapper unifica las tiendas duplicadas
Se toman filas de validación de las tiendas 57, 58 y 10, se les cambia el `shop_id` por el ID duplicado (0, 1 y 11) y se comprueba que el pipeline devuelve exactamente el mismo resultado.

In [6]:
originales = X_valid[X_valid["shop_id"].isin([57, 58, 10])].groupby("shop_id").head(2)
duplicadas = originales.copy()
duplicadas["shop_id"] = duplicadas["shop_id"].replace({57: 0, 58: 1, 10: 11}).astype("int8")

print("shop_id antes:", originales["shop_id"].tolist(), "->", duplicadas["shop_id"].tolist())
print("Después del Mapper:", pasos["unificar_tiendas"].transform(duplicadas)["shop_id"].tolist())
iguales = np.allclose(preprocesamiento.transform(originales), preprocesamiento.transform(duplicadas))
print("¿El pipeline produce la misma salida con los IDs duplicados?:", iguales)

shop_id antes: [57, 57, 58, 58, 10, 10] -> [0, 0, 1, 1, 11, 11]
Después del Mapper: [57, 57, 58, 58, 10, 10]
¿El pipeline produce la misma salida con los IDs duplicados?: True


### 5.b Transformación de validación y de test
Los datos de validación (meses 27 a 33) y de test (mes 34) se transforman con los parámetros aprendidos en el entrenamiento, sin volver a ajustar.

In [7]:
X_valid_t = preprocesamiento.transform(X_valid)
X_test_t = preprocesamiento.transform(X_test)
for nombre, Xt in [("Validación", X_valid_t), ("Test", X_test_t)]:
    print(f"{nombre:<11} {Xt.shape[0]:>9,} filas x {Xt.shape[1]} columnas | NaN: {Xt.isna().sum().sum()} | "
          f"mismas columnas que el entrenamiento: {Xt.columns.equals(X_train_t.columns)}")

# Categorías no vistas en el entrenamiento: el OrdinalEncoder les asigna -1 (se revisa antes del escalado)
hasta_codificar = preprocesamiento[: list(pasos).index("codificar_ordinal") + 1]
for nombre, X in [("Validación", X_valid), ("Test", X_test)]:
    codificado = hasta_codificar.transform(X)
    no_vistas = {c: int((codificado[c] == -1).sum()) for c in codigos}
    print(f"Filas con categoría no vista en {nombre}: {no_vistas}")

print("\nMedia de algunas variables transformadas (en el entrenamiento es 0 por construcción):")
cols = ["item_cnt_month_lag_1", "item_avg_cnt_lag_1", "item_price_last", "item_age", "date_block_num"]
display(pd.DataFrame({"entrenamiento": X_train_t[cols].mean(), "validación": X_valid_t[cols].mean(), "test": X_test_t[cols].mean()}))

Validación  1,614,364 filas x 24 columnas | NaN: 0 | mismas columnas que el entrenamiento: True
Test          214,200 filas x 24 columnas | NaN: 0 | mismas columnas que el entrenamiento: True


Filas con categoría no vista en Validación: {'shop_id': 5413, 'item_category_id': 3833, 'shop_city': 0, 'item_cat_subtype': 0}
Filas con categoría no vista en Test: {'shop_id': 5100, 'item_category_id': 630, 'shop_city': 0, 'item_cat_subtype': 0}

Media de algunas variables transformadas (en el entrenamiento es 0 por construcción):


,entrenamiento,validación,test
item_cnt_month_lag_1,-0.0000,-0.0322,-0.0379
item_avg_cnt_lag_1,0.0000,-0.0451,-0.0424
item_price_last,0.0000,0.1728,0.3163
item_age,-0.0000,0.5519,0.5922
date_block_num,0.0000,2.5987,3.5451


**Interpretación de las verificaciones:**

- **Mapper:** las filas con los IDs duplicados (0, 1 y 11) se convierten en 57, 58 y 10, y el pipeline produce exactamente la misma salida que con los IDs originales. La unificación de tiendas del EDA queda incorporada en el `.pkl`.
- **Validación y test** se transforman sin errores, con las mismas 24 columnas y sin NaN.
- **Categorías no vistas:** 5,100 filas de test (las 5,100 de la **tienda 36**) y 630 de categorías de producto que no aparecen en los meses 12 a 26 reciben el código -1. Según el EDA, la tienda 36 solo tiene ventas en octubre de 2015 (mes 33), por eso no existe en este entrenamiento. Esto se corrige en el notebook 04: el pipeline final se reentrena con los meses 12 a 33, que sí incluyen a la tienda 36. En validación ocurre lo mismo con algunas tiendas que abrieron después del mes 26.
- **Cambio de distribución:** en el entrenamiento las variables escaladas tienen media 0; en validación y test, `date_block_num` tiene medias de 2.6 y 3.5 (son meses posteriores), `item_age` es mayor (el catálogo envejece) y los lags son un poco menores (las ventas caen). Es el comportamiento esperado de una serie con tendencia decreciente, y es la razón por la que la validación debe ser secuencial y no aleatoria.

## 6. Guardado del pipeline en `.pkl`

In [8]:
ruta = MODELS_DIR / "preprocessing_pipeline.pkl"
joblib.dump(preprocesamiento, ruta)
print(f"Pipeline guardado en {ruta} ({ruta.stat().st_size / 1e3:,.1f} KB)")

# Verificación: el pipeline cargado desde disco produce exactamente la misma salida
cargado = joblib.load(ruta)
muestra = X_valid.sample(50_000, random_state=0)
print("¿El pipeline cargado reproduce la transformación?:",
      np.array_equal(cargado.transform(muestra).to_numpy(), preprocesamiento.transform(muestra).to_numpy()))
print("Pasos:", [nombre for nombre, _ in cargado.steps])

Pipeline guardado en ..\models\preprocessing_pipeline.pkl (8.2 KB)


¿El pipeline cargado reproduce la transformación?: True
Pasos: ['unificar_tiendas', 'imputar_lags', 'imputar_precio', 'imputar_categoricas', 'codificar_ordinal', 'codificar_onehot', 'eliminar_item_id', 'outliers_precio', 'log_precio', 'log_conteos', 'escalar', 'float32']


**Conclusión:** el pipeline de preprocesamiento, con 12 pasos (el `Mapper` de clase, 10 operadores de `feature-engine`, uno de los cuales envuelve el `StandardScaler` de sklearn, y el paso `DowncastFloat32`), quedó ajustado con los meses 12 a 26 y guardado en **`models/preprocessing_pipeline.pkl`** (8.2 KB, porque solo almacena parámetros: medianas, modas, diccionarios de codificación, límites, medias y desviaciones). Al cargarlo desde disco reproduce exactamente la misma transformación.

En el notebook 04 este pipeline transforma el entrenamiento y la validación para comparar 15 modelos, y después se reentrena, junto con el modelo ganador como paso final, con los meses 12 a 33.